In [1]:
import pandas as pd
import numpy as np
import re 
import geopandas as gpd

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import collections
import os

In [2]:
data_input_path = "../../data/final/raw_data/"
shapefiles_input_path = "../../data/final/shapefiles/"

data_output_path = "../../data/final/keywords_and_location_names/"

# 1. Scraping location names from Wikipedia

The file contains wikipedia URLs for locations on admin level 0, 1 and 2 for Iraq, Jordan, Lebanon, Palestine and Syria.

This file was manually created using among others the following overview tables (September 2024):
- https://en.wikipedia.org/wiki/Governorates_of_Iraq
- https://en.wikipedia.org/wiki/Governorates_of_Jordan
- https://en.wikipedia.org/wiki/Governorates_of_Lebanon
- https://en.wikipedia.org/wiki/Governorates_of_Palestine
- https://en.wikipedia.org/wiki/Governorates_of_Syria

- https://en.wikipedia.org/wiki/Districts_of_Iraq
- https://en.wikipedia.org/wiki/Districts_of_Jordan
- https://en.wikipedia.org/wiki/Districts_of_Lebanon
- https://en.wikipedia.org/wiki/Districts_of_Syria

In [3]:
url_list = pd.read_csv(data_input_path + "wikipedia_url_list.csv")

In [4]:
def extract_titles_and_bold_names(url):
    
    """
    Extracts the title in English, the title in Arabic (if available), 
    and the bold names mentioned in the first two paragraphs of a Wikipedia article for a given URL.
    
    Parameters:
    url (str): The URL of the webpage to extract information from.
    Returns:
    tuple: A tuple containing the title in English, the title in Arabic (if available), and a list of bold names found in the webpage.
    """
    
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Extract the title in English
    title_eng = soup.find('h1', {'id': 'firstHeading'}).text

    # Find the first two paragraphs
    paragraphs = soup.find_all('p', limit=2)
    
    # Extract bold names from both paragraphs
    bold_names = []
    for paragraph in paragraphs:
        bold_names.extend([b.text for b in paragraph.find_all(['b', 'strong'])])

    
    # Extract the title in Arabic if the page has an Arabic version
    title_ara = None
    lang_link = soup.find('a', {'hreflang': 'ar'})
    if lang_link:
        href = lang_link['href']
        if href.startswith('http'):
            ara_url = href
        else:
            ara_url = f"https:{href}"

        ara_response = requests.get(ara_url)
        ara_soup = BeautifulSoup(ara_response.content, 'html.parser')
        title_ara = ara_soup.find('h1', {'id': 'firstHeading'}).text

    return title_eng, title_ara, bold_names

In [5]:
wiki_urls = []
wiki_titles_eng = []
wiki_titles_ara = []
adm_levels = []
countries = []
bold_names_list = []

# Iterate over the URLs and create a table with all the extracted informaton.
for url, adm, country in tqdm(zip(url_list["uri"], url_list["adm"], url_list["country"])):
    title_eng, title_ara, bold_names = extract_titles_and_bold_names(url)
    wiki_urls.append(url)
    wiki_titles_eng.append(title_eng)
    wiki_titles_ara.append(title_ara)
    bold_names_list.append(bold_names)
    adm_levels.append(adm)
    countries.append(country)

wiki_df = pd.DataFrame({"name": wiki_titles_eng, "name_ara": wiki_titles_ara, "bold_names": bold_names_list, "url": wiki_urls, "adm": adm_levels, "country": countries})

395it [06:24,  1.03it/s]


## 1.1 Clean bold names

In [6]:
# Create a dictionary with the index of the dataframe as index and a list containing the name and bold names as value
wiki_dict = {}

for row in wiki_df.iterrows():
    # Use the english name column and the bold names column to create a list as dictionary entry
    wiki_dict[row[0]] = np.array(row[1]["name"])
    wiki_dict[row[0]] = np.append(wiki_dict[row[0]], row[1]["bold_names"])

In [7]:
# Only keep strings containing any characters a-zA-Z
for key in wiki_dict.keys():
    delete_indices = []
    for idx, word in enumerate(wiki_dict[key]):
        if not bool(re.search('[a-zA-Z]', word)):
            delete_indices.append(idx)
    for i in sorted(delete_indices, reverse=True):
            wiki_dict[key] = np.delete(wiki_dict[key], i)  

In [8]:
# Create function that removes prefixes for Arabic articles from the given list of words
def remove_prefix(list):
    """
    Converts all words in the given list to lowercase and removes Arabic article prefixes such as "al-", "al ", "ar-", "ar ", "as-", "as ", "el-", "el ", "ath-", "ath ", "az-", and "az ".
    
    Args:
        list (list): The list of words to be processed.
    
    Returns:
        list: The processed list of words with lowercase and cleaned prefixes.
    """
    
    return [word.removeprefix("al-").removeprefix("al ").removeprefix("ar-").removeprefix("ar ").removeprefix("as-").removeprefix("as ").removeprefix("el-").removeprefix("el ").removeprefix("ath-").removeprefix("ath ").removeprefix("az-").removeprefix("az ").removeprefix("an-").removeprefix("an ") for word in list]

In [9]:
# Iterate over the dictionary and apply the following cleaning steps:
for key in wiki_dict.keys():
    
    # Create lower case version of the dictionary 
    wiki_dict[key] = [word.lower() for word in wiki_dict[key]]
    
    # Replace multiple whitespace characters with a single whitespace character
    wiki_dict[key] = [word.replace("  ", " ") for word in wiki_dict[key]]
    
    # Exclude district, province and governorate and department from the name list
    wiki_dict[key] = [word.replace("district", "") for word in wiki_dict[key]]
    wiki_dict[key] = [word.replace("province", "") for word in wiki_dict[key]]
    wiki_dict[key] = [word.replace("governorate", "") for word in wiki_dict[key]]
    wiki_dict[key] = [word.replace("department", "") for word in wiki_dict[key]]
    wiki_dict[key] = [word.replace("markaz", "") for word in wiki_dict[key]]
    
    # Exclude leading or tailing whitespaces
    wiki_dict[key] = [word.strip() for word in wiki_dict[key]]
        
    # Exclude empty strings
    wiki_dict[key] = [word for word in wiki_dict[key] if word != ""]
    
    # Remove Arabic articles at the beginning of each name
    wiki_dict[key] = remove_prefix(wiki_dict[key])
    
    # Delete duplicates from each name list
    wiki_dict[key] = np.unique(wiki_dict[key])    

In [10]:
# Check if elements in each name list are superset of other elements of the same list. If so, remove the superset
# Example: "new york city" is a superset of "new york". Hence, we delete "new york city" from the name list.

# Iterate over name lists in the dictionary
for key in wiki_dict.keys():
    
    # Create a list containing the single words (split by " ") of each name list
    lists = []
    
    # Go through the entries of the list
    for name in wiki_dict[key]:
        
        # Split each entry by whitespace (creating a list of lists)
        lists.append(name.split(" "))
    
    # Check for each list in the list of lists if it is a subset of another list. If so, remove the superset
    for list1 in lists:
        for list2 in lists:
            if (set(list1) < set(list2)) & (list1 in lists):
                print(f"subset {list1}     -     superset: {list2} (removed)")
                lists.remove(list2)
    
    # Of the remaining elements in the list, join them to a single string and append them to a new list
    clean_lists = []
    for list in lists:
        list = " ".join(list)
        clean_lists.append(list)
    
    wiki_dict[key] = clean_lists # Replace the previous lists with the lists cleaned from word supersets

subset ['iraq']     -     superset: ['republic', 'of', 'iraq'] (removed)
subset ['jordan']     -     superset: ['hashemite', 'kingdom', 'of', 'jordan'] (removed)
subset ['lebanon']     -     superset: ['republic', 'of', 'lebanon'] (removed)
subset ['palestine']     -     superset: ['state', 'of', 'palestine'] (removed)
subset ['palestine']     -     superset: ['historic', 'palestine'] (removed)
subset ['palestine']     -     superset: ['palestine', '(region)'] (removed)
subset ['palestinian', 'authority']     -     superset: ['palestinian', 'national', 'authority'] (removed)
subset ['palestinian', 'authority']     -     superset: ['palestinian', 'national', 'authority'] (removed)
subset ['palestinian', 'territories']     -     superset: ['occupied', 'palestinian', 'territories'] (removed)
subset ['palestinian', 'territories']     -     superset: ['occupied', 'palestinian', 'territories'] (removed)
subset ['palestine']     -     superset: ['state', 'of', 'palestine'] (removed)
subset ['

In [11]:
df_list = []
for key in wiki_dict.keys():    
    
    # Get the row from the wiki_df dataframe that corresponds to the dictionary key
    df_row = wiki_df.iloc[[key]]
    
    # Repeat this row for the length of the list with unique names stored in the dictionary and add the unique names as a new column
    df_row_repeated = pd.DataFrame(np.repeat(df_row.values, len(wiki_dict[key]), axis=0))
    df_row_repeated.columns = wiki_df.columns
    df_row_repeated["name_unique"] = wiki_dict[key]
    
    # Append the repeated dataframe to the list of dataframes
    df_list.append(df_row_repeated)
    
# Concatenate the list dataframes that are repeated rows with the unique name column to a single dataframe
wiki_unique_df = pd.concat(df_list).reset_index(drop=True)

## 1.2 Manual cleaning of the dataframe

### 1.2.1 English

In [12]:
# 1. Delete rows with the following names
wiki_unique_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] != "other reasons this message may be displayed:"]
wiki_unique_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] != "major insurgent attacks"]
wiki_unique_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] != "qasr, karak"]
wiki_unique_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] != "rashid, baghdad"]
wiki_unique_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] != "at taibah , irbid"]
wiki_unique_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] != "el"]
wiki_unique_df.reset_index(drop=True, inplace=True)

In [13]:
# 2. Correct names in the following cases 
wiki_unique_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] != "9 nissan",]
wiki_unique_df.loc[wiki_unique_df["name_unique"] == "7 nissan", "name_unique"] = "nissan"

wiki_unique_df.loc[wiki_unique_df["name_unique"] == "ruseifa(h)", "name_unique"] = "ruseifa"
ruseifa_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] == "ruseifa",].copy()
ruseifa_df["name_unique"] = "ruseifah"

wiki_unique_df.loc[wiki_unique_df["name_unique"] == "rusaifa(h)", "name_unique"] = "rusaifa"
rusaifa_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] == "rusaifa",].copy()
rusaifa_df["name_unique"] = "rusaifah"

wiki_unique_df.loc[wiki_unique_df["name_unique"] == "mharda, also mahardah or mhardeh", "name_unique"] = "mahardah"
mahardah_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] == "mahardah", ].copy()
mahardah_df["name_unique"] = "mhardeh"

amedi_df = wiki_unique_df.loc[wiki_unique_df["name_unique"] == "amedi",].copy()
amedi_df["name_unique"] = "amadiya"

wiki_unique_df = pd.concat([wiki_unique_df, ruseifa_df, rusaifa_df,mahardah_df, amedi_df]).reset_index(drop=True)

In [14]:
# 3. Delete the following String patterns
wiki_unique_df["name_unique"] = wiki_unique_df["name_unique"].str.replace(", jordan", "")
wiki_unique_df["name_unique"] = wiki_unique_df["name_unique"].str.replace(", iraq", "")
wiki_unique_df["name_unique"] = wiki_unique_df["name_unique"].str.replace(" or", "")

In [15]:
# 4. Rename the following names
wiki_unique_df.loc[wiki_unique_df["name_unique"] == "north", "name_unique"] = "north lebanon"
wiki_unique_df.loc[wiki_unique_df["name_unique"] == "south", "name_unique"] = "south lebanon"

### 1.2.2 Arabic

In [16]:
# Create a dictionary to count the number of times a word appears in the name_ara column
word_count = {}
for name_ara in wiki_unique_df.loc[~wiki_unique_df["name_ara"].isna(), "name_ara"].values:
    for word in name_ara.split(" "):
        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

- 140 قضاء - an administrative unit of territory, used in the Asiatic part of the Arab world (https://en.wiktionary.org/wiki/%D9%82%D8%B6%D8%A7%D8%A1#Noun)
- 70 منطقة - zone, vicinity, range, district, area, territory, sphere (https://en.wiktionary.org/wiki/%D9%85%D9%86%D8%B7%D9%82%D8%A9)
- 64 محافظة - a governorate; an administrative division of state territory (https://en.wiktionary.org/wiki/%D9%85%D8%AD%D8%A7%D9%81%D8%B8%D8%A9)
- 34 لواء - province, district (https://en.wiktionary.org/wiki/%D9%84%D9%88%D8%A7%D8%A1)
- 20 (محافظة) - a governorate; an administrative division of state territory (https://en.wiktionary.org/wiki/%D9%85%D8%AD%D8%A7%D9%81%D8%B8%D8%A9)
- 13 مركز - capital, chief town of a province (https://en.wiktionary.org/wiki/%D9%85%D8%B1%D9%83%D8%B2)
- محافظة - a governorate; an administrative division of state territory
- العاصمة Capital city (https://en.wikipedia.org/wiki/Capital_city)

In [17]:
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.replace("قضاء", "")
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.replace("منطقة", "")
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.replace("(محافظة)", "")
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.replace("محافظة", "")
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.replace("لواء", "")
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.replace("مركز", "")
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.replace("العاصمة", "")

In [18]:
wiki_unique_df.loc[(wiki_unique_df["name_unique"] == "amman") & (wiki_unique_df["adm"] == 1), "name_ara"] = "عمان"
wiki_unique_df.loc[(wiki_unique_df["name_unique"] == "muhafazat al-asima") & (wiki_unique_df["adm"] == 1), "name_ara"] = "عمان"

Replace everything between parenthesis as this mostly is the location on a higher administrative level

In [19]:
# Remove brackets and their content from the name_ara column
wiki_unique_df["name_ara"] = wiki_unique_df["name_ara"].str.replace("()", "")

# Remove brackets and their content from the name_ara column
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()) & (wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.contains("\(")),"name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()) & (wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()),"name_ara"].str.contains("\(")),"name_ara"].str.replace(r'\(.*?\)', '', regex=True).str.strip()

# Strip leading and trailing whitespaces from Arabic names
wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()), "name_ara"] = wiki_unique_df.loc[(~wiki_unique_df["name_ara"].isna()), "name_ara"].str.strip()

## 1.4 Save wiki_df

In [20]:
wiki_unique_df.drop(columns=["name", "bold_names"], inplace=True)
wiki_unique_df.rename(columns={"name_unique": "location_name", "name_ara" : "location_name_ara", "url":"uri"}, inplace=True)

In [21]:
wiki_unique_df["country"] = wiki_unique_df["country"].str.lower()

In [22]:
wiki_unique_df = wiki_unique_df.drop_duplicates()

In [23]:
wiki_unique_df.sort_values(by=["adm", "country", "location_name"], inplace=True)
wiki_unique_df = wiki_unique_df.reset_index(drop=True)

In [24]:
# Save the cleaned dataframe to a CSV file
wiki_unique_df.to_csv(data_input_path + "mashreq_wiki_clean.csv", index=False)

# 2. Merging scraped Wikipedia names with the shapefile

In [25]:
wiki_unique_df = pd.read_csv(data_input_path + "mashreq_wiki_clean.csv")

# Loading shapefile
wb_shp = gpd.read_file(shapefiles_input_path + "WB_clean/WB_clean.shp", index=False)

/home/dominik/Documents/wb2024/implementations/food_crises_mashreq/exploratory_news_sources/.venv/lib/python3.10/site-packages/pyogrio/raw.py:196: RuntimeWarning: driver ESRI Shapefile does not support open option INDEX
  return ogr_read(


In [26]:
wb_shp["NAME"] = wb_shp["NAME"].str.replace("markaz", "").str.strip()

## 2.1 Admin Level 1

In [27]:
# Location names on level 1 that appear in the WB shapefile but not in the Wikipedia data
english_additional_names_from_wb_shp = {

# "wbshp" : "wiki_df"

# Iraq
"basrah" : "basra",
"dahuk" : "duhok",
"kerbala" : "karbala",
"missan" : "maysan",
"ninewa" : "ninawa",
"qadissiya" : "qadisiyah",
"salah al-din" : "salah ad din",
"thi-qar" : "dhi qar",
"wassit" : "wasit",
"arbil" : "erbil",
"salah ad-din" : "salah ad din",

# Jordan
"ajloon" : "ajloun",
"jarash" : "jerash",
"tafiela" : "tafilah",
"ma`an" : "ma'an",

# Syria
"ḥasakah" : "hasakah",
"raqqah" : "raqqa",
"suwaydā'" : "suwayda",
"dar`ā" : "daraa",
"dayr az zawr" : "deir ez-zor",
"ḥimṣ" : "homs",
"ţarţūs" : "tartus",
"hasakeh" : "hasakah",
"sweida" : "suwayda",
"dar'a" : "daraa",
"deir-ez-zor" : "deir ez-zor",
"idleb" : "idlib",
"lattakia" : "latakia",
"rural damascus" : "rif dimashq",

# Lebanon
"bekaa" : "beqaa",
"nabatiye" : "nabatieh",
"nabatiyah" : "nabatieh"}

In [28]:
new_rows = []
for key, value in zip (english_additional_names_from_wb_shp.keys(), english_additional_names_from_wb_shp.values()):
    
    # Create a copy of the existing row based on the dictionary value and replace the name with the new name based on the dictionary key
    new_row = wiki_unique_df.loc[(wiki_unique_df["adm"] == 1) & (wiki_unique_df["location_name"] == value),].copy()
    new_row["location_name"] = key
    
    # Append the new row to the list of new rows
    new_rows.append(new_row)

In [29]:
# Add the new rows to the existing rows
total_rows = new_rows + [wiki_unique_df]
wiki_unique_df = pd.concat(total_rows).reset_index(drop=True)

In [30]:
# Location names on level 1 that appear in the WB shapefile but not in the Wikipedia data
arabic_additional_names_from_wb_shp = {
    
# Iraq
"الانبار" : "anbar",
"اربيل" : "erbil",

# Syria
"مدينة دمشق" : "damascus",

# Lebanon
"بعلبك - الهرمل" : "baalbek-hermel",

}

In [31]:
new_rows = []
for key, value in zip (arabic_additional_names_from_wb_shp.keys(), arabic_additional_names_from_wb_shp.values()):
    
    # Create a copy of the existing row based on the dictionary value and replace the name with the new name based on the dictionary key
    new_row = wiki_unique_df.loc[(wiki_unique_df["adm"] == 1) & (wiki_unique_df["location_name"] == value),].copy()
    new_row["location_name_ara"] = key
    
    # Append the new row to the list of new rows
    new_rows.append(new_row)

In [32]:
# Add the new rows to the existing rows
total_rows = new_rows + [wiki_unique_df]
wiki_unique_df = pd.concat(total_rows).reset_index(drop=True)

## 2.2 Admin level 2

Check how many wb_shp English location names adml 2 are not in the wiki_unique_df

In [33]:
wiki_df_iraq_districts = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "iraq"),"location_name"].values
wb_shp_iraq_districts = wb_shp.loc[(wb_shp["NAM_0"] == "iraq") & (~wb_shp["NAM_2"].isna()), "NAM_2"].values

wiki_df_jordan_districts = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "jordan"),"location_name"].values
wb_shp_jordan_districts = wb_shp.loc[(wb_shp["NAM_0"] == "jordan") & (~wb_shp["NAM_2"].isna()), "NAM_2"].values

wiki_df_lebanon_districts = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "lebanon"),"location_name"].values
wb_shp_lebanon_districts = wb_shp.loc[(wb_shp["NAM_0"] == "lebanon") & (~wb_shp["NAM_2"].isna()), "NAM_2"].values

wiki_df_palestine_districts = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "palestine"),"location_name"].values
wb_shp_palestine_districts = wb_shp.loc[(wb_shp["NAM_0"] == "palestine") & (~wb_shp["NAM_2"].isna()), "NAM_2"].values

wiki_df_syria_districts = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "syria"),"location_name"].values
wb_shp_syria_districts = wb_shp.loc[(wb_shp["NAM_0"] == "syria") & (~wb_shp["NAM_2_AL1"].isna()), "NAM_2_AL1"].values

In [34]:
# Iraq district names that appear in the wiki_df but not in the World Bank shapefile
#sorted(wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "iraq") & (~wiki_unique_df["location_name"].isin(wb_shp_iraq_districts)),"location_name"].values)

# Jordan district names that appear in the wiki_df but not in the World Bank shapefile
#sorted(wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "jordan") & (~wiki_unique_df["location_name"].isin(wb_shp_jordan_districts)),"location_name"].values)

# Lebanon district names that appear in the wiki_df but not in the World Bank shapefile
#sorted(wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "lebanon") & (~wiki_unique_df["location_name"].isin(wb_shp_lebanon_districts)),"location_name"].values)

# Palestine district names that appear in the wiki_df but not in the World Bank shapefile
#sorted(wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "palestine") & (~wiki_unique_df["location_name"].isin(wb_shp_palestine_districts)),"location_name"].values)

# Syria district names that appear in the wiki_df but not in the World Bank shapefile
#sorted(wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["country"] == "syria") & (~wiki_unique_df["location_name"].isin(wb_shp_syria_districts)),"location_name"].values)

In [35]:
# Iraq district names that appear in the World Bank shapefile but not in the wiki_df
#wb_shp.loc[(~wb_shp["NAM_2"].isna()) & (wb_shp["ID_0"] == "iq") & (~wb_shp["NAM_2"].isin(wiki_df_iraq_districts)), ["NAM_2", "NAM_2_AL1", "NAM_2_AL2"]]

# Jordan district names that appear in the World Bank shapefile but not in the wiki_df
#wb_shp.loc[(~wb_shp["NAM_2"].isna()) & (wb_shp["ID_0"] == "jo") & (~wb_shp["NAM_2"].isin(wiki_df_jordan_districts)), ["NAM_2", "NAM_2_AL1", "NAM_2_AL2"]]

# Lebanon district names that appear in the World Bank shapefile but not in the wiki_df
#wb_shp.loc[(~wb_shp["NAM_2"].isna()) & (wb_shp["ID_0"] == "lb") & (~wb_shp["NAM_2"].isin(wiki_df_lebanon_districts)), ["NAM_2", "NAM_2_AL1", "NAM_2_AL2"]]

# Palestine district names that appear in the World Bank shapefile but not in the wiki_df
#wb_shp.loc[(~wb_shp["NAM_2"].isna()) & (wb_shp["ID_0"] == "ps") & (~wb_shp["NAM_2"].isin(wiki_df_palestine_districts)), ["NAM_2", "NAM_2_AL1", "NAM_2_AL2"]]

# Syria district names that appear in the World Bank shapefile but not in the wiki_df
#wb_shp.loc[(~wb_shp["NAM_2"].isna()) & (wb_shp["ID_0"] == "sy") & (~wb_shp["NAM_2"].isin(wiki_df_syria_districts)), ["NAM_2", "NAM_2_AL1", "NAM_2_AL2"]]

In [36]:
iraq_districts_english_additional_names_from_wb_shp = {
    
# "wb_shp" : "wiki_df"
'adhamiya' : 'adhamiyah',
'ana' : 'anah',
'azezia' : 'aziziyah',
'baladrooz' : 'balad ruz',
'dabes' : 'dibis',
'dahuk' : 'duhok',
'darbandihkan' : 'darbandikhan',
'falluja' : 'fallujah',
'fao' : 'faw',
'fares' : 'faris',
'hawiga' : 'hawija',
'heet' : 'hit',
"ka'im" : "qa'im",
'kadhmiyah' : 'kadhimiya',
'koisnjaq' : 'koy sanjaq',
'mahmoudiya' : 'mahmudiya',
'mejar al-kabi' : 'mejar al-kabir',
'mergasur' : 'mergasor',
"na'maniya" : "nu'maniya",
'nassriya' : 'nasiriyah',
'penjwin' : 'penjwen',
'rania' : 'ranya',
'resafa' : 'rusafa',
'shikhan' : 'shekhan',
'telafar' : 'tel afar',
'thawra 1' : 'thawra',
'thawra 2' : 'thawra',
"ra'ua" : 'rawah', 
'shatt al-arab' : None,
'sumel' : None,
'thethar' : None,
'tilkaif' : 'tel keif',
'mejar kabi' : 'mejar al-kabir',
}

In [37]:
new_rows = []
for key, value in zip (iraq_districts_english_additional_names_from_wb_shp.keys(), iraq_districts_english_additional_names_from_wb_shp.values()):
    
    # Create a copy of the existing row based on the dictionary value and replace the name with the new name based on the dictionary key
    new_row = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["location_name"] == value),].copy()
    new_row["location_name"] = key
    
    # Append the new row to the list of new rows
    new_rows.append(new_row)
    
# Add the new rows to the existing rows
total_rows = new_rows + [wiki_unique_df]
wiki_unique_df = pd.concat(total_rows).reset_index(drop=True)

In [38]:
jordan_districts_english_additional_names_from_wb_shp = {
    
'aghwar al-janoobiya' : 'āghwār al-janūbī',
'aghwar al-shimaliya' : 'āghwār ash-shamāliyah',
'ail' : None,
'ajloon' : "qaṣabah 'ajlūn",
'aqaba' : "qaṣabah al-'aqabah",
'ardha' : None,
'ayy' : "'ayy",
"bal'ama" : None,
'beni kinana' : 'bani kinanah',
'beni obaid' : "banī 'obeīd",
'bsaira' : 'bṣaīrā',
'dair ala' : 'deir alla',
'dhilail' : None,
"faqooa'" : "faqū'e", # less sure
'hashimiya' : 'hāshimiyah',
'ira & yarqa' : None,
'irbid' :  'qaṣabah irbid',
'jafer' : None,
"jami'ah" : None,
'jarash' : 'qaṣabah jarash',
'jeeza' : 'jizah',
'karak' : 'qaṣabah al-karak',
'koora' : 'kourah',
'kufranjeh' : 'kufranjah',
"ma'an" : "qaṣabah ma'ān",
'mafraq' : 'mafraq qasabah',
'mahes & fhais' : 'fuheis', # less sure
'mazar al-janoobi' : 'mazār al-janūbī',
'mazar al-shimali' : 'mazār ash-shamālī',
'mraigha' : None,
'muwaqar' : 'muwaqqar',
"na'oor" : "na'our",
'qaser' : 'qasr',
'qwaira' : 'qūaīrah',
'qwaismeh' : 'quesmah', # less sure
'ruwaished' : 'ruwaishid',
'sabha' : None,
'sama al-serhan' : None,
'shobak' : 'shoubak',
'shoona al-janoobiya' : 'ash-shunah al-janubiyah',
'tafiela' : 'qaṣabah aṭ-ṭafīlah',
'teeba' : None,
'theeban' : 'aṭ-ṭaībah', # less sure
'wadi araba' : None,
'wadi moosa' : None,
'wadisseer' : 'wadi al-seer',
'wasatiya' : 'wasṭīyah',
'zay' : None,
'aghwar janoobiya' : 'āghwār al-janūbī',
'aghwar shimaliya' : 'āghwār ash-shamāliyah',
'mazar janoobi' : 'mazār al-janūbī',
'mazar shimali' : 'mazār ash-shamālī',
'shoona janoobiya' : 'ash-shunah al-janubiyah'
}

In [39]:
new_rows = []
for key, value in zip (jordan_districts_english_additional_names_from_wb_shp.keys(), jordan_districts_english_additional_names_from_wb_shp.values()):
    
    # Create a copy of the existing row based on the dictionary value and replace the name with the new name based on the dictionary key
    new_row = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["location_name"] == value),].copy()
    new_row["location_name"] = key
    
    # Append the new row to the list of new rows
    new_rows.append(new_row)
    
# Add the new rows to the existing rows
total_rows = new_rows + [wiki_unique_df]
wiki_unique_df = pd.concat(total_rows).reset_index(drop=True)

In [40]:
lebanon_districts_english_additional_names_from_wb_shp = {
    
'bint jubail' : 'bint jbeil',
'hasbaiya' : 'hasbaya',
'jubail': 'jbeil',
'kasrouane' : "keserwan",
'marjaayoun' : 'marjeyoun',
'minieh-danieh' : 'miniyeh-danniyeh',
'nabatiye' : 'nabatieh',
'rachiaya' : 'rashaya',
'saida' : "sidon",
'sour' : "tyre",
'west bekaa' : 'western beqaa',
'zahle' : 'zahlé'

}

In [41]:
new_rows = []
for key, value in zip (lebanon_districts_english_additional_names_from_wb_shp.keys(), lebanon_districts_english_additional_names_from_wb_shp.values()):
    
    # Create a copy of the existing row based on the dictionary value and replace the name with the new name based on the dictionary key
    new_row = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["location_name"] == value),].copy()
    new_row["location_name"] = key
    
    # Append the new row to the list of new rows
    new_rows.append(new_row)
    
# Add the new rows to the existing rows
total_rows = new_rows + [wiki_unique_df]
wiki_unique_df = pd.concat(total_rows).reset_index(drop=True)

In [42]:
palestine_districts_english_additional_names_from_wb_shp = {
    
'ariha' : 'jericho', 
'deir al balah' : 'deir al-balah', 
'jabalya' : 'north gaza', 
'khalil' : 'hebron', 
'ramallah' : 'ramallah and al-bireh',
'quds' : "jerusalem"

}

In [43]:
new_rows = []
for key, value in zip (palestine_districts_english_additional_names_from_wb_shp.keys(), palestine_districts_english_additional_names_from_wb_shp.values()):
    
    # Create a copy of the existing row based on the dictionary value and replace the name with the new name based on the dictionary key
    new_row = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["location_name"] == value),].copy()
    new_row["location_name"] = key
    
    # Append the new row to the list of new rows
    new_rows.append(new_row)
    
# Add the new rows to the existing rows
total_rows = new_rows + [wiki_unique_df]
wiki_unique_df = pd.concat(total_rows).reset_index(drop=True)

In [44]:
syria_districts_english_additional_names_from_wb_shp = {
    
"a'zaz" : 'azaz',
'abu-kamal' : 'abu kamal',
'ash-shaykh badr' : 'shaykh badr',
'at-tall' : 'tall',
"ayn al-'arab" : '`ain al-`arab',
'duma' : 'douma',
'fiq zawiyah' : "fiq",
'harim' : 'harem',
"jabal sam'an" : None,
'jebel saman' : None,
'jablah' : 'jableh',
'jisr ash-shughur' : 'jisr ash-shugur',
"ma'arrat an-nu'man" : "ma'arrat nu'man",
'madinah dimashq' : "damascus",
'al-hasakah' : 'hasakah',
'al-ladhiqiyah' : 'latakia', # less sure
'al-qunaytrah' : 'qunaytirah',
'ar-raqqah' : 'raqqa',
'as-suwayda' : 'suwayda',
"dar'a" : 'daraa',
'dayr az-zawr' : 'deir ez-zor',
'hamah' : 'hama',
'hims' : 'homs',
'idlib' : 'idlib',
'tartus' : 'tartus',
'muhradah' : 'mahardah',
'nabk' : 'nabek',
'qardahah' : 'qardaha',
"ra's-al-ayn" : 'ras al-ayn',
'safirah' : 'safira',
'saqlabiyah' :  'suqaylabiyah',
"shahba'" :  'shahba',
'tall abyad' : 'tell abyad',
'tall kalakh' : 'talkalakh',
'yabrud' : 'yabroud',
    
'ain al arab' : '`ain al-`arab',
'at tall' : "tall",
'banyas' : 'baniyas',
"dar'a" : 'daraa',
'deir-ez-zor' : 'deir ez-zor',
'dreikish' : 'duraykish',
'haffa' : 'haffah',
'hasakeh' : 'hasakah',
'idleb' : 'idlib',
'jarablus' : 'jarabulus',
'jisr-ash-shugur' : 'jisr ash-shugur',
'lattakia' : 'latakia',
"ma'ra" : "ma'arrat nu'man",
'makhrim' : "mukharram",
'malikeyyeh' : 'malikiyah',
'menbij' : 'manbij',
'quamishli' : 'qamishli',
'ras al ain' : 'ras al-ayn',
'rural damascus' : "rif dimashq",
'salamiyeh' : 'salamiyah',
'sheikh badr' : 'shaykh badr',
'sweida' : 'suwayda',
'tadmor' : 'tadmur',
'tartous' : 'tartus',
'tell abiad' : 'tell abyad',
'zabdani' : 'zabadani' 
    
}

In [45]:
new_rows = []
for key, value in zip (syria_districts_english_additional_names_from_wb_shp.keys(), syria_districts_english_additional_names_from_wb_shp.values()):
    
    # Create a copy of the existing row based on the dictionary value and replace the name with the new name based on the dictionary key
    new_row = wiki_unique_df.loc[(wiki_unique_df["adm"] == 2) & (wiki_unique_df["location_name"] == value),].copy()
    new_row["location_name"] = key
    
    # Append the new row to the list of new rows
    new_rows.append(new_row)
    
# Add the new rows to the existing rows
total_rows = new_rows + [wiki_unique_df]
wiki_unique_df = pd.concat(total_rows).reset_index(drop=True)

In [46]:
wiki_unique_df.sort_values(by=["adm", "country", "location_name"], inplace=True)
wiki_unique_df.drop_duplicates(inplace=True)
wiki_unique_df.reset_index(drop=True, inplace=True)

## 2.4 Merge with World Bank shapefile

In [47]:
wb_shp.loc[wb_shp["NAME"] == "north", "NAME"] = "north lebanon"
wb_shp.loc[wb_shp["NAME"] == "south", "NAME"] = "south lebanon"

In [48]:
df = pd.merge(wiki_unique_df, wb_shp, left_on=["adm", "country", "location_name"], right_on=["adml", "NAM_0", "NAME"], how='outer', indicator='Exist')

In [49]:
df["Exist"].value_counts()

Exist
both          320
left_only     287
right_only     18
Name: count, dtype: int64

In [50]:
# In the case of Sadr_City, there are two places called Thawra 1 and Thawra 2 in the World Bank shapefile
# For the code to work, I will replace the uri of thawra 2 and later bring it back to the original value
df.loc[df["ID"] == "iq_bg_10", "uri"] = None

for uri in df["uri"].unique():
    if len(df.loc[(df["uri"] == uri) & (~df["ID"].isna()), "ID"].unique()) == 1:
        uri_id = df.loc[(df["uri"] == uri) & (~df["ID"].isna()), "ID"].unique()[0]
        df.loc[(df["uri"] == uri), "ID"] = uri_id

df.loc[df["ID"] == "iq_bg_10", "uri"] = "https://en.wikipedia.org/wiki/Sadr_City"

In [51]:
# Associate IDs with locations
df.loc[(df["adm"] == 0), "adml"] = 0
df.loc[(df["country"] == "palestine") & (df["adm"] == 0), "ID"] = "ps"

In [52]:
# Assign IDs to the locations from wiki_df which have a uri but no ID
country_iso_dict = {"iraq" : "iq",
                    "jordan" : "jo",
                    "lebanon" : "lb",
                    "syria" : "sy",
                    "palestine" : "ps"}

uri_id_dict = {}
# Per country iterate through the uris of locations with no id and associate an id to each uri
for country in df.loc[~df["country"].isna(), "country"].unique():
    uris = df.loc[(df["ID"].isna()) & (~df["uri"].isna()) & (df["country"] == country), "uri"].unique()
    for idx, uri in enumerate(uris):
        uri_id_dict[uri] = f"{country_iso_dict[country]}_00_{idx}" # 00 indicates that this location is not represented in the shapefile
        
# Assign the IDs to the locations based on the URIs
for key in uri_id_dict.keys():
    df.loc[df["uri"] == key, "ID"] = uri_id_dict[key]

# 3. Create Output

## 3.1 Query specifications table for NewsAPI

In [53]:
newsapi_qurey_file = wiki_unique_df.copy()
newsapi_qurey_file.loc[newsapi_qurey_file["adm"] == 2, "location_name"] = np.nan
newsapi_qurey_file.loc[newsapi_qurey_file["adm"] == 2, "location_name_ara"] = np.nan

In [54]:
newsapi_qurey_file = newsapi_qurey_file.drop_duplicates()
newsapi_qurey_file.reset_index(drop=True, inplace=True)

In [55]:
newsapi_qurey_file = newsapi_qurey_file[['location_name', 'uri', 'location_name_ara', 'country', 'adm']]

In [56]:
newsapi_qurey_file.to_csv(data_output_path + "newsapi_query_specifications.csv", index=False)

## 3.2 Name:ID dictionaries

### 3.2.1 Create English Name:ID dictionary

In [57]:
import pickle

In [58]:
# Create function that removes prefixes for Arabic articles from the given list of words
def remove_prefix(list):
    """
    Converts all words in the given list to lowercase and removes Arabic article prefixes such as "al-", "al ", "ar-", "ar ", "as-", "as ", "el-", "el ", "ath-", "ath ", "az-", and "az ".
    
    Args:
        list (list): The list of words to be processed.
    
    Returns:
        list: The processed list of words with lowercase and cleaned prefixes.
    """
    
    return [word.removeprefix("al-").removeprefix("al ").removeprefix("ar-").removeprefix("ar ").removeprefix("as-").removeprefix("as ").removeprefix("el-").removeprefix("el ").removeprefix("ath-").removeprefix("ath ").removeprefix("az-").removeprefix("az ").removeprefix("an-").removeprefix("an ") for word in list]

In [59]:
english_name_dict = {}

for id in df["ID"].unique():
    english_names = []
    english_names.append(df.loc[(df["ID"] == id) & (~df["location_name"].isna()), "location_name"].unique())
    english_names.append(df.loc[(df["ID"] == id) & (~df["NAME"].isna()), "NAME"].unique())
    
    if len(id.split("_")) == 2:
        english_names.append(df.loc[(df["ID"] == id) & (~df["NAM_1_AL1"].isna()), "NAM_1_AL1"].unique())
        english_names.append(df.loc[(df["ID"] == id) & (~df["NAM_1_AL2"].isna()), "NAM_1_AL2"].unique())
        
    if len(id.split("_")) == 3:
        english_names.append(df.loc[(df["ID"] == id) & (~df["NAM_2_AL1"].isna()), "NAM_2_AL1"].unique())
        english_names.append(df.loc[(df["ID"] == id) & (~df["NAM_2_AL2"].isna()), "NAM_2_AL2"].unique())
        
    english_names_no_prefix = []
    for name in english_names:
        english_names_no_prefix.append(remove_prefix(name))        
            
    english_name_dict[id] = np.unique(np.concatenate(english_names_no_prefix))

In [60]:
assert len([id for id in wb_shp["ID"] if id not in english_name_dict.keys()]) == 0, "Not all IDs of the shapefile are represented in the English name dictionary."

In [61]:
# Check if elements in each name list are superset of other elements of the same list. If so, remove the superset
# Example: "new york city" is a superset of "new york". Hence, we delete "new york city" from the name list.

# Iterate over name lists in the dictionary
for key in english_name_dict.keys():
    
    # Create a list containing the single words (split by " ") of each name list
    lists = []
    
    # Go through the entries of the list
    for name in english_name_dict[key]:
        
        # Split each entry by whitespace (creating a list of lists)
        lists.append(name.split(" "))
    
    # Check for each list in the list of lists if it is a subset of another list. If so, remove the superset
    for list1 in lists:
        for list2 in lists:
            if (set(list1) < set(list2)) & (list1 in lists):
                print(f"subset {list1}     -     superset: {list2} (removed)")
                lists.remove(list2)
    
    # Of the remaining elements in the list, join them to a single string and append them to a new list
    clean_lists = []
    for list in lists:
        list = " ".join(list)
        clean_lists.append(list)
    
    english_name_dict[key] = clean_lists # Replace the previous lists with the lists cleaned from word supersets

subset ['palestine']     -     superset: ['state', 'of', 'palestine'] (removed)
subset ['nissan']     -     superset: ['tisa', 'nissan'] (removed)
subset ['thawra']     -     superset: ['thawra', '1'] (removed)
subset ['irbid']     -     superset: ['qaṣabah', 'irbid'] (removed)
subset ['jarash']     -     superset: ['qaṣabah', 'jarash'] (removed)
subset ['mafraq']     -     superset: ['mafraq', 'qasabah'] (removed)
subset ['ariha']     -     superset: ['ariha', '(jericho)'] (removed)
subset ['khalil']     -     superset: ['khalil', '(hebron)'] (removed)
subset ['quds']     -     superset: ['quds', '(jerusalem)'] (removed)
subset ['ramallah']     -     superset: ['ramallah', 'and', 'al-bireh'] (removed)
subset ['tall']     -     superset: ['at', 'tall'] (removed)
subset ['fiq']     -     superset: ['fiq', 'zawiyah'] (removed)


In [62]:
assert len([id for id in wb_shp["ID"] if id not in english_name_dict.keys()]) == 0, "The dictionary does not contain the names of all IDs of the shapefile."

In [63]:
english_name_dict

{'iq': ['iraq'],
 'jo': ['jordan'],
 'lb': ['lebanon'],
 'ps': ['occupied palestinian territory',
  'palestine',
  'palestinian authority',
  'palestinian territories'],
 'sy': ['syria', 'syrian arab republic'],
 'iq_an': ['anbar'],
 'iq_ar': ['arbil', 'erbil'],
 'iq_bb': ['babil', 'babylon'],
 'iq_bg': ['baghdad'],
 'iq_ba': ['basra', 'basrah'],
 'iq_da': ['dahuk', 'duhok'],
 'iq_dq': ['dhi qar', 'thi-qar'],
 'iq_qa': ['diwaniyah', 'qadisiyah', 'qadissiya', 'qādisiyyah'],
 'iq_di': ['diyala'],
 'iq_ka': ['karbala', "karbala'", 'kerbala'],
 'iq_ts': ['kirkuk'],
 'iq_ma': ['maysan', 'missan'],
 'iq_mu': ['muthanna'],
 'iq_na': ['najaf'],
 'iq_ni': ['ninawa', 'nineveh', 'ninewa'],
 'iq_sd': ['saladin', 'salah ad din', 'salah ad-din', 'salah al-din'],
 'iq_sl': ['sulaymaniyah'],
 'iq_wa': ['wasit', 'wassit'],
 'jo_aj': ['ajloon', 'ajloun', 'ajlun'],
 'jo_am': ['amman', 'muhafazat al-asima'],
 'jo_aq': ['aqaba'],
 'jo_ba': ['balqa', "balqa'"],
 'jo_ir': ['irbed', 'irbid'],
 'jo_ja': ['jara

In [64]:
file_path = data_output_path + 'id_english_location_name.pkl'

with open(file_path, 'wb') as f:
    pickle.dump(english_name_dict, f)

### 3.2.2 Create Arabic Name:ID dictionary

In [65]:
arabic_name_dict = {}

for id in df["ID"].unique():
    arabic_names = []
    arabic_names.append(df.loc[(df["ID"] == id) & (~df["location_name_ara"].isna()), "location_name_ara"].unique())
    arabic_names.append(df.loc[(df["ID"] == id) & (~df["NAME_NTVE"].isna()), "NAME_NTVE"].unique())
    
    arabic_name_dict[id] = np.unique(np.concatenate(arabic_names))

In [66]:
file_path = data_output_path + 'id_arabic_location_name.pkl'

with open(file_path, 'wb') as f:
    pickle.dump(arabic_name_dict, f)